<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/09_main_llm_annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from pathlib import Path
import json
import hashlib

import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load frozen partitions and previously verified annotation protocol.
with (BASE / "configs/holdout_partition_manifest_v1.json").open() as f:
    manifest = json.load(f)

with (BASE / "configs/annotation_protocol_v1.json").open() as f:
    protocol = json.load(f)

with (BASE / "configs/llm_pilot_manifest_v2.json").open() as f:
    pilot_config = json.load(f)

assert manifest["manifest_version"] == "v1"

MODEL_ID = pilot_config["model_id"]
MODEL_REVISION = pilot_config["model_revision_sha"]
SYSTEM_PROMPT = protocol["system_prompt"]

assert MODEL_ID == "meta-llama/Llama-3.1-8B-Instruct"
assert MODEL_REVISION == (
    "0e9e39f249a16976918f6564b8830bc894c89659"
)

prompt_hash = hashlib.sha256(
    SYSTEM_PROMPT.encode("utf-8")
).hexdigest()

assert prompt_hash == pilot_config["prompt_sha256"]

# Load only identifiers and raw article text; never load reference labels.
source = pd.read_csv(
    BASE / "data/splits/noisy_training.csv",
    usecols=["record_id", "text"]
)

assert len(source) == 16200
assert source["record_id"].is_unique
assert source["text"].notna().all()

def fingerprint_ids(ids):
    payload = "\n".join(
        str(int(value)) for value in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def fingerprint_text(frame):
    digest = hashlib.sha256()
    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

annotation_frames = []

for name, expected_count in [
    ("training", 13621),
    ("holdout", 2000)
]:
    specification = manifest["partitions"][name]
    ids = set(map(int, specification["record_ids"]))

    frame = source.loc[
        source["record_id"].isin(ids)
    ].copy()

    assert len(ids) == len(frame) == expected_count
    assert fingerprint_ids(ids) == specification["id_sha256"]
    assert fingerprint_text(frame) == specification["raw_text_sha256"]

    frame["partition"] = name
    annotation_frames.append(frame)

annotation_inputs = pd.concat(
    annotation_frames,
    ignore_index=True
)

assert len(annotation_inputs) == 15621
assert annotation_inputs["record_id"].is_unique
assert torch.cuda.is_available(), "Enable GPU in Colab Runtime settings."

print("MAIN LLM ANNOTATION — INPUT VALIDATION")
print("Training:", 13621)
print("Holdout:", 2000)
print("Total annotation inputs:", len(annotation_inputs))
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Prompt SHA256:", prompt_hash)
print("GPU:", torch.cuda.get_device_name(0))
print("Frozen manifest: VERIFIED")
print("Raw text integrity: VERIFIED")
print("Reference labels loaded: NO")
print("Inference started: NO")


Mounted at /content/drive
MAIN LLM ANNOTATION — INPUT VALIDATION
Training: 13621
Holdout: 2000
Total annotation inputs: 15621
Model: meta-llama/Llama-3.1-8B-Instruct
Revision: 0e9e39f249a16976918f6564b8830bc894c89659
Prompt SHA256: 77a8d266f6f9d3e2dcdbeb906182e0d64f0836190fe84617a638d3ab3c5ed794
GPU: Tesla T4
Frozen manifest: VERIFIED
Raw text integrity: VERIFIED
Reference labels loaded: NO
Inference started: NO


In [2]:

%pip install -q "transformers==5.20.1" "huggingface_hub==1.33.0" "bitsandbytes==0.50.2" "accelerate"

import torch
import transformers
import bitsandbytes
import importlib.metadata as metadata

from google.colab import userdata
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Verify the environment and model access.
assert torch.cuda.is_available()
assert transformers.__version__ == "5.20.1"
assert metadata.version("bitsandbytes") == "0.50.2"
assert len(annotation_inputs) == 15621
assert MODEL_REVISION == pilot_config["model_revision_sha"]

hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Add HF_TOKEN in Colab Secrets."

# Reproduce the pilot's 4-bit NF4 configuration.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

if "model" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=hf_token
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=hf_token,
        quantization_config=quantization_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True
    )

model.eval()

assert next(model.parameters()).device.type == "cuda"
assert model.config._name_or_path == MODEL_ID

print("MAIN LLM MODEL LOADING — PASSED")
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", metadata.version("bitsandbytes"))
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Allocated VRAM:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print("Model mode: EVALUATION")
print("Inference completed: NO")


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

MAIN LLM MODEL LOADING — PASSED
Model: meta-llama/Llama-3.1-8B-Instruct
Revision: 0e9e39f249a16976918f6564b8830bc894c89659
Transformers: 5.20.1
BitsAndBytes: 0.50.2
GPU: Tesla T4
Allocated VRAM: 5.32 GB
Model mode: EVALUATION
Inference completed: NO


In [3]:

import json
import hashlib
import os
import torch
import pandas as pd

OUTPUT_DIR = BASE / "outputs" / "llm_main_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

smoke_path = OUTPUT_DIR / "smoke_test_v1.json"

# Select records from both frozen partitions.
smoke_df = pd.concat([
    annotation_inputs.loc[
        annotation_inputs["partition"] == partition
    ].head(4)
    for partition in ("training", "holdout")
], ignore_index=True)

assert len(smoke_df) == 8
assert smoke_df["record_id"].is_unique
assert smoke_df["partition"].value_counts().to_dict() == {
    "training": 4,
    "holdout": 4
}

def parse_annotation(response):
    try:
        result = json.loads(response.strip())
        if (
            isinstance(result, dict)
            and set(result) == {"label"}
            and type(result["label"]) is int
            and result["label"] in (0, 1, 2, 3)
        ):
            return result["label"]
    except (ValueError, TypeError, AttributeError):
        pass
    return None

def text_sha256(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()

# Reuse an existing checkpoint only after integrity verification.
if smoke_path.exists():
    with smoke_path.open(encoding="utf-8") as f:
        checkpoint = json.load(f)

    assert checkpoint["model_id"] == MODEL_ID
    assert checkpoint["model_revision_sha"] == MODEL_REVISION
    assert checkpoint["prompt_sha256"] == prompt_hash

    predictions = checkpoint["predictions"]
    assert len(predictions) == 8

    for row, result in zip(
        smoke_df.itertuples(index=False), predictions
    ):
        assert result["record_id"] == int(row.record_id)
        assert result["partition"] == row.partition
        assert result["text_sha256"] == text_sha256(row.text)
        assert result["predicted_label"] == parse_annotation(
            result["raw_response"]
        )

    checkpoint_status = "EXISTING_VERIFIED"

else:
    predictions = []
    model.eval()

    for row in smoke_df.itertuples(index=False):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Classify this news:\n\n{row.text}"
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = output[0][
            inputs["input_ids"].shape[-1]:
        ]

        response = tokenizer.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        predictions.append({
            "record_id": int(row.record_id),
            "partition": row.partition,
            "text_sha256": text_sha256(row.text),
            "raw_response": response,
            "predicted_label": parse_annotation(response)
        })

    checkpoint = {
        "experiment_id": "s_slte_main_annotation_v1",
        "model_id": MODEL_ID,
        "model_revision_sha": MODEL_REVISION,
        "prompt_sha256": prompt_hash,
        "generation": {
            "do_sample": False,
            "max_new_tokens": 32
        },
        "predictions": predictions
    }

    temporary_path = smoke_path.with_suffix(".json.tmp")

    with temporary_path.open("w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2)

    os.replace(temporary_path, smoke_path)
    checkpoint_status = "NEW_CHECKPOINT_SAVED"

# Verify the persisted results.
with smoke_path.open(encoding="utf-8") as f:
    verified = json.load(f)

assert verified == checkpoint

results_df = pd.DataFrame(predictions)
valid_count = int(results_df["predicted_label"].notna().sum())

print("MAIN LLM ANNOTATION — SMOKE TEST")
print("Checkpoint:", checkpoint_status)
print("Total records:", len(results_df))
print("Valid predictions:", valid_count)
print("Invalid predictions:", len(results_df) - valid_count)

print("\nPREDICTION SUMMARY")
display(
    results_df[
        ["record_id", "partition", "predicted_label"]
    ]
)

print("\nVALIDATION")
print("Model revision: VERIFIED")
print("Prompt fingerprint: VERIFIED")
print("Reference labels accessed: NO")
print("Checkpoint saved:", smoke_path)
print(
    "Status:",
    "PASSED" if valid_count == 8 else "INVALID_OUTPUT_REVIEW"
)


MAIN LLM ANNOTATION — SMOKE TEST
Checkpoint: NEW_CHECKPOINT_SAVED
Total records: 8
Valid predictions: 8
Invalid predictions: 0

PREDICTION SUMMARY


,record_id,partition,predicted_label
0,11,training,0
1,14,training,2
2,19,training,2
3,44,training,0
4,167,holdout,3
5,234,holdout,3
6,378,holdout,2
7,407,holdout,3



VALIDATION
Model revision: VERIFIED
Prompt fingerprint: VERIFIED
Reference labels accessed: NO
Checkpoint saved: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_main_v1/smoke_test_v1.json
Status: PASSED


In [4]:

import json
import os
import time
import torch
import pandas as pd

# Define the first reproducible production batch.
checkpoint_dir = OUTPUT_DIR / "records"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

batch_df = pd.concat([
    annotation_inputs.loc[
        annotation_inputs["partition"] == partition
    ].head(50)
    for partition in ("training", "holdout")
], ignore_index=True)

assert len(batch_df) == 100
assert batch_df["record_id"].is_unique
assert batch_df["partition"].value_counts().to_dict() == {
    "training": 50,
    "holdout": 50
}

# Recover the verified smoke-test predictions.
with smoke_path.open(encoding="utf-8") as f:
    smoke_checkpoint = json.load(f)

assert smoke_checkpoint["model_revision_sha"] == MODEL_REVISION
assert smoke_checkpoint["prompt_sha256"] == prompt_hash
assert smoke_checkpoint["generation"] == {
    "do_sample": False,
    "max_new_tokens": 32
}

smoke_predictions = {
    int(item["record_id"]): item
    for item in smoke_checkpoint["predictions"]
}

assert len(smoke_predictions) == 8
assert set(smoke_predictions).issubset(
    set(batch_df["record_id"].astype(int))
)

def verify_checkpoint(result, row):
    assert result["record_id"] == int(row.record_id)
    assert result["partition"] == row.partition
    assert result["text_sha256"] == text_sha256(row.text)
    assert result["model_id"] == MODEL_ID
    assert result["model_revision_sha"] == MODEL_REVISION
    assert result["prompt_sha256"] == prompt_hash
    assert result["generation"] == {
        "do_sample": False,
        "max_new_tokens": 32
    }
    assert result["predicted_label"] == parse_annotation(
        result["raw_response"]
    )
    assert result["is_valid"] == (
        result["predicted_label"] is not None
    )

model.eval()

records = []
new_count = 0
smoke_reused = 0
existing_reused = 0

start_time = time.perf_counter()

for index, row in enumerate(
    batch_df.itertuples(index=False), start=1
):
    record_id = int(row.record_id)
    path = checkpoint_dir / f"record_{record_id}.json"

    if path.exists():
        with path.open(encoding="utf-8") as f:
            result = json.load(f)

        existing_reused += 1

    elif record_id in smoke_predictions:
        saved = smoke_predictions[record_id]

        assert saved["partition"] == row.partition
        assert saved["text_sha256"] == text_sha256(row.text)

        result = {
            **saved,
            "model_id": MODEL_ID,
            "model_revision_sha": MODEL_REVISION,
            "prompt_sha256": prompt_hash,
            "generation": {
                "do_sample": False,
                "max_new_tokens": 32
            },
            "is_valid": saved["predicted_label"] is not None,
            "source": "verified_smoke_test"
        }

        smoke_reused += 1

    else:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Classify this news:\n\n{row.text}"
            }
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = output[0][
            inputs["input_ids"].shape[-1]:
        ]

        response = tokenizer.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        prediction = parse_annotation(response)

        result = {
            "record_id": record_id,
            "partition": row.partition,
            "text_sha256": text_sha256(row.text),
            "model_id": MODEL_ID,
            "model_revision_sha": MODEL_REVISION,
            "prompt_sha256": prompt_hash,
            "generation": {
                "do_sample": False,
                "max_new_tokens": 32
            },
            "raw_response": response,
            "predicted_label": prediction,
            "is_valid": prediction is not None,
            "source": "main_inference"
        }

        new_count += 1

    # Validate before persisting each immutable checkpoint.
    verify_checkpoint(result, row)

    if not path.exists():
        temporary_path = path.with_suffix(".json.tmp")

        with temporary_path.open("w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        os.replace(temporary_path, path)

    records.append(result)

    if index % 20 == 0:
        print(f"Processed: {index}/100")

elapsed = time.perf_counter() - start_time

results_df = pd.DataFrame(records)

assert len(results_df) == 100
assert results_df["record_id"].is_unique

valid_count = int(results_df["is_valid"].sum())

print("\nMAIN ANNOTATION — PRODUCTION BATCH 1")
print("Total records:", len(results_df))
print("Valid predictions:", valid_count)
print("Invalid predictions:", len(results_df) - valid_count)
print("Newly inferred:", new_count)
print("Smoke predictions reused:", smoke_reused)
print("Existing checkpoints reused:", existing_reused)
print("Batch duration (seconds):", round(elapsed, 2))

print("\nLABEL DISTRIBUTION BY PARTITION")
display(
    results_df.groupby(
        ["partition", "predicted_label"],
        dropna=False
    ).size().reset_index(name="count")
)

print("\nVALIDATION")
print("Model revision: VERIFIED")
print("Prompt fingerprint: VERIFIED")
print("Reference labels accessed: NO")
print("Saved checkpoint records:", len(records))
print("Checkpoint directory:", checkpoint_dir)
print(
    "Status:",
    "PASSED" if valid_count == 100 else "INVALID_OUTPUTS_DETECTED"
)


Processed: 20/100
Processed: 40/100
Processed: 60/100
Processed: 80/100
Processed: 100/100

MAIN ANNOTATION — PRODUCTION BATCH 1
Total records: 100
Valid predictions: 100
Invalid predictions: 0
Newly inferred: 92
Smoke predictions reused: 8
Existing checkpoints reused: 0
Batch duration (seconds): 87.91

LABEL DISTRIBUTION BY PARTITION


,partition,predicted_label,count
0,holdout,0,16
1,holdout,1,7
2,holdout,2,17
3,holdout,3,10
4,training,0,8
5,training,1,1
6,training,2,21
7,training,3,20



VALIDATION
Model revision: VERIFIED
Prompt fingerprint: VERIFIED
Reference labels accessed: NO
Saved checkpoint records: 100
Checkpoint directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_main_v1/records
Status: PASSED


In [5]:

import json
import os
import time
import hashlib
from pathlib import Path

import pandas as pd
import torch

# Process a bounded number of new predictions per execution.
MAX_NEW_PER_RUN = 2000

checkpoint_dir = OUTPUT_DIR / "records"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

expected_ids = set(annotation_inputs["record_id"].astype(int))
assert len(expected_ids) == 15621
assert annotation_inputs["record_id"].is_unique

# Freeze the main inference configuration separately from the pilot.
main_manifest = {
    "experiment_id": "s_slte_main_annotation_v1",
    "partition_manifest_version": "v1",
    "training_id_sha256": manifest["partitions"]["training"]["id_sha256"],
    "holdout_id_sha256": manifest["partitions"]["holdout"]["id_sha256"],
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "prompt_sha256": prompt_hash,
    "quantization": "4-bit NF4",
    "text_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "generation": {
        "do_sample": False,
        "max_new_tokens": 32
    },
    "expected_records": 15621
}

main_manifest_path = OUTPUT_DIR / "main_annotation_manifest_v1.json"

if main_manifest_path.exists():
    with main_manifest_path.open(encoding="utf-8") as f:
        assert json.load(f) == main_manifest
else:
    with main_manifest_path.open("x", encoding="utf-8") as f:
        json.dump(main_manifest, f, indent=2)

# Detect unexpected checkpoints before annotation.
existing_paths = list(checkpoint_dir.glob("record_*.json"))
existing_ids = {int(p.stem.split("_")[-1]) for p in existing_paths}

assert len(existing_ids) == len(existing_paths)
assert existing_ids.issubset(expected_ids)

def annotate_record(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Classify this news:\n\n{row.text}"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    prediction = parse_annotation(response)

    result = {
        "record_id": int(row.record_id),
        "partition": row.partition,
        "text_sha256": text_sha256(row.text),
        "model_id": MODEL_ID,
        "model_revision_sha": MODEL_REVISION,
        "prompt_sha256": prompt_hash,
        "generation": {
            "do_sample": False,
            "max_new_tokens": 32
        },
        "raw_response": response,
        "predicted_label": prediction,
        "is_valid": prediction is not None,
        "source": "main_inference"
    }

    del inputs, output, generated
    return result

model.eval()

new_count = 0
reused_count = 0
verified_count = 0
invalid_ids = []
start_time = time.perf_counter()

# Process frozen records in deterministic partition order.
for row in annotation_inputs.itertuples(index=False):
    record_id = int(row.record_id)
    path = checkpoint_dir / f"record_{record_id}.json"

    if path.exists():
        with path.open(encoding="utf-8") as f:
            result = json.load(f)

        verify_checkpoint(result, row)
        reused_count += 1

    else:
        if new_count >= MAX_NEW_PER_RUN:
            break

        result = annotate_record(row)
        verify_checkpoint(result, row)

        # Write a complete checkpoint before moving to another record.
        temporary_path = path.with_suffix(".json.tmp")

        with temporary_path.open("w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        os.replace(temporary_path, path)
        new_count += 1

    verified_count += 1

    if not result["is_valid"]:
        invalid_ids.append(record_id)

    if new_count > 0 and new_count % 250 == 0:
        if not path.exists():
            raise RuntimeError("Checkpoint missing after saving.")

        if result["source"] == "main_inference":
            print(
                f"Newly annotated: {new_count}/{MAX_NEW_PER_RUN} "
                f"| Current record ID: {record_id}"
            )

elapsed = time.perf_counter() - start_time

# Inspect the current persisted checkpoint inventory.
saved_files = list(checkpoint_dir.glob("record_*.json"))
saved_ids = {int(p.stem.split("_")[-1]) for p in saved_files}

assert len(saved_files) == len(saved_ids)
assert saved_ids.issubset(expected_ids)

completed = len(saved_ids)
remaining = len(expected_ids - saved_ids)

print("\nMAIN LLM ANNOTATION — PRODUCTION PROGRESS")
print("Expected records:", len(expected_ids))
print("Saved checkpoints:", completed)
print("Remaining records:", remaining)
print("Completion:", f"{completed / len(expected_ids):.2%}")

print("\nCURRENT EXECUTION")
print("New predictions:", new_count)
print("Previously saved and verified:", reused_count)
print("Records verified this execution:", verified_count)
print("Invalid outputs encountered:", len(invalid_ids))
print("Execution duration (seconds):", round(elapsed, 2))

if invalid_ids:
    print("Invalid record IDs:", invalid_ids[:20])

print("\nINTEGRITY")
print("Frozen partition manifest: VERIFIED")
print("Annotation configuration: VERIFIED")
print("Model revision: VERIFIED")
print("Reference labels accessed: NO")
print("Checkpoint directory:", checkpoint_dir)

print(
    "STATUS:",
    "ALL RECORDS CHECKPOINTED" if remaining == 0
    else "CHECKPOINTED — RERUN SAME CELL TO CONTINUE"
)


Newly annotated: 250/2000 | Current record ID: 2995
Newly annotated: 500/2000 | Current record ID: 5207
Newly annotated: 750/2000 | Current record ID: 7100
Newly annotated: 1000/2000 | Current record ID: 9365
Newly annotated: 1250/2000 | Current record ID: 11389
Newly annotated: 1500/2000 | Current record ID: 13538
Newly annotated: 1750/2000 | Current record ID: 15749
Newly annotated: 2000/2000 | Current record ID: 17974

MAIN LLM ANNOTATION — PRODUCTION PROGRESS
Expected records: 15621
Saved checkpoints: 2100
Remaining records: 13521
Completion: 13.44%

CURRENT EXECUTION
New predictions: 2000
Previously saved and verified: 50
Records verified this execution: 2050
Invalid outputs encountered: 0
Execution duration (seconds): 1483.19

INTEGRITY
Frozen partition manifest: VERIFIED
Annotation configuration: VERIFIED
Model revision: VERIFIED
Reference labels accessed: NO
Checkpoint directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_main_v1/records
STATUS: CHECKPOINTED — RERUN SAME 